# Expense Claim Analysis

This notebook demonstrates how to create agents that use plugins to process travel expenses from local receipt images, generate an expense claim email, and visualize expense data using a pie chart. Agents dynamically choose functions based on the task context.

Steps:
1. OCR Agent processes the local receipt image and extracts travel expense data.
2. Email Agent generates an expense claim email.

### Halimbawa ng isang senaryo ng gastusin sa biyahe:
Isipin mong isa kang empleyadong naglalakbay para sa isang pulong pang-negosyo sa ibang lungsod. Mayroon ang iyong kumpanya ng patakaran na bayaran ang lahat ng makatwirang gastusin na may kaugnayan sa paglalakbay. Narito ang pagkakahati-hati ng potensyal na mga gastusin sa biyahe:
- Transportasyon:
Pamasahe sa eroplano para sa isang round trip mula sa iyong lungsod patungo sa lungsod ng destinasyon.
Taxi o mga serbisyo ng ride-hailing papunta at pabalik sa paliparan.
Lokal na transportasyon sa lungsod ng destinasyon (tulad ng pampublikong transportasyon, mga inuupahang sasakyan, o taxi).

- Panuluyan:
Panunuluyan sa hotel ng tatlong gabi sa isang mid-range na business hotel malapit sa lugar ng pulong.

- Pagkain:
Araw-araw na allowance para sa pagkain ng almusal, tanghalian, at hapunan, batay sa patakaran ng kumpanya sa per diem.

- Iba pang mga Gastusin:
Bayad sa paradahan sa paliparan.
Singil sa internet sa hotel.
Tip o maliliit na singil sa serbisyo.

- Dokumentasyon:
Isusumite mo ang lahat ng resibo (lipad, taxi, hotel, pagkain, atbp.) at ang isang nakumpletong expense report para sa reimbursement.


## Import mga kinakailangang library

I-import ang mga kinakailangang library at mga module para sa notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Define Expense Models

 Gumawa ng Pydantic model para sa mga indibidwal na gastusin at isang ExpenseFormatter class upang i-convert ang user query sa istrukturadong data ng gastusin.

 Ang bawat gastusin ay ire-representa sa format na:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - Paggawa ng Email

Lumikha ng isang tool function upang gumawa ng email para sa pagsusumite ng claim sa gastusin.
- Ginagamit ng tool na ito ang `@tool` decorator mula sa Microsoft Agent Framework.
- Kinakalkula nito ang kabuuang halaga ng mga gastusin at inaayos ang mga detalye sa katawan ng email.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Tool para sa Pagkuha ng Gastusin sa Paglalakbay mula sa mga Larawan ng Resibo

Gumawa ng tool na function upang kunin ang mga gastusin sa paglalakbay mula sa mga larawan ng resibo.
- Ginagamit ng tool na ito ang `@tool` decorator mula sa Microsoft Agent Framework.
- Binabasa nito ang larawan ng resibo, ini-encode ito bilang base64, at ibinabalik ang data URI para suriin ng ahente.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Pagpoproseso ng Mga Gastusin

Tukuyin ang mga ahente at ikonekta ang mga ito sa isang sunud-sunod na workflow gamit ang `WorkflowBuilder`.
- Kinukuha ng OCR agent ang nakaayos na datos ng gastusin mula sa larawan ng resibo gamit ang tool na `load_receipt_image`.
- Kinukuha ng Email agent ang nakuha na datos at bumubuo ng isang propesyonal na email para sa pag-angkin ng gastusin gamit ang tool na `generate_expense_email`.
- Gumagamit ang `WorkflowBuilder` ng `add_edge` upang lumikha ng isang sunud-sunod na pipeline: OCR Agent → Email Agent.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Pangunahing function

Buuin ang sunud-sunod na workflow at patakbuhin ito upang iproseso ang larawan ng resibo at gumawa ng expense claim email.


> **Tandaan:** Ang workflow na ito ay kasalukuyang nagpapasa ng larawan ng resibo bilang base64 na teksto, na karamihan sa mga chat model (kabilang ang gpt-4o) ay hindi ituturing bilang isang larawan.
> Maaari rin nitong lumagpas sa context window ng modelo. Mas mainam na patakbuhin ang OCR gamit ang Azure AI Vision (o ibang OCR na tool) at ipasa lamang ang na-extract na teksto, o baguhin para ipadala ang larawan bilang isang `image_url` na mensahe.
> Kung nais mo lamang iwasan ang mga error sa context, subukan ang mas maliit na larawan ng resibo o isang modelo na may mas malaking context window.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
